In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

class SincLayer(nn.Module):
    def __init__(self, out_channels):
        super(SincLayer, self).__init__()
        self.out_channels = out_channels
        # Parámetros aprendibles (frecuencias) para la explicación ante-hoc
        self.f1 = nn.Parameter(torch.linspace(30, 4000, out_channels))
        self.f2 = nn.Parameter(torch.linspace(100, 8000, out_channels))

    def forward(self, x):
        # Si x es [N, 16000], le falta la dimensión de canales. La añadimos:
        if x.dim() == 2:
            x = x.unsqueeze(1) # Ahora x es [N, 1, 16000]
            
        # narrow(dimensión, inicio, longitud)
        # Recortamos en la dimensión 2 (Tiempo/Samples)
        return torch.narrow(x, 2, 0, self.out_channels)

class SincNetClassifier(nn.Module):
    def __init__(self, out_channels=20):
        super(SincNetClassifier, self).__init__()
        self.sinc = SincLayer(out_channels=out_channels)
        self.fc = nn.Linear(out_channels, 2)

    def forward(self, x):
        # x: [Batch, 1, Longitud]
        x = self.sinc(x)             # Salida: [Batch, 1, out_channels]
        x = x.reshape(x.size(0), -1)  # Aplanamos a [Batch, out_channels]
        return self.fc(x)

# Instanciar el modelo
model = SincNetClassifier(out_channels=20)

In [6]:
# Verificación y corrección de dimensiones de entrada
if X_raw.dim() == 2:
    print(f"Dimensiones originales: {X_raw.shape}. Corrigiendo a 3D...")
    X_raw = X_raw.unsqueeze(1) 
    
print(f"Forma final de entrada para el modelo: {X_raw.shape}") # Debe ser [N, 1, 16000]

# Configuración del entrenamiento
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(21):
    model.train()
    optimizer.zero_grad()
    
    # Inferencia
    outputs = model(X_raw) 
    
    loss = criterion(outputs, y_raw)
    loss.backward()
    optimizer.step()
    
    if epoch % 5 == 0:
        print(f"Época {epoch} | Loss: {loss.item():.4f}")

print("¡Entrenamiento completado sin errores de dimensiones!")

Dimensiones originales: torch.Size([0, 1]). Corrigiendo a 3D...
Forma final de entrada para el modelo: torch.Size([0, 1, 1])


RuntimeError: start (0) + length (20) exceeds dimension size (1).